In [ ]:
# !pip install 'git+https://github.com/facebookresearch/detectron2.git'

In [ ]:
# import some common libraries
import albumentations as A
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
from sklearn.metrics import jaccard_score
from tqdm.notebook import tqdm

%matplotlib inline
import copy
import csv
import datetime
import json
import os
import random
import urllib.request

import cv2

# import some common detectron2 utilities
import detectron2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.config.config import CfgNode as CN
from detectron2.data import (
    DatasetCatalog,
    MetadataCatalog,
    build_detection_test_loader,
    build_detection_train_loader,
)
from detectron2.data import detection_utils as utils
from detectron2.engine import DefaultPredictor, DefaultTrainer, launch
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.structures import Boxes, BoxMode, Instances
from detectron2.structures.boxes import pairwise_iou
from detectron2.utils.logger import setup_logger
from detectron2.utils.visualizer import ColorMode, GenericMask, Visualizer
from torch.autograd import Variable

# import some common pytorch utilities
from torch.utils.data import DataLoader, Dataset

setup_logger()

In [ ]:
# Define the location of current directory, which should contain train, test, and train.json.
BASE_DIR = "./"
OUTPUT_DIR = "{}/output".format(BASE_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Object Detection

### Data Loader

In [ ]:
category = {"tap": 0, "flange": 1, "bend": 2, "tee": 3}


def get_detection_data():
    data_dirs = "{}/".format(BASE_DIR)
    json_file = os.path.join(data_dirs, "test.json")
    with open(json_file) as f:
        imgs_anns = json.load(f)

    dataset = []
    for idx, v in enumerate(imgs_anns):
        record = {}
        project_id = list(v["projects"])[0]
        filename = os.path.join(data_dirs, "test", v["data_row"]["external_id"])
        height, width = v["media_attributes"]["height"], v["media_attributes"]["width"]
        annotations = v["projects"][project_id]["labels"][0]["annotations"]["objects"]
        record["file_name"] = filename
        record["image_id"] = idx
        record["height"] = height
        record["width"] = width
        record["annotations"] = [
            {
                "bbox": [
                    obj["bounding_box"]["left"],
                    obj["bounding_box"]["top"],
                    obj["bounding_box"]["width"],
                    obj["bounding_box"]["height"],
                ],
                "bbox_mode": BoxMode.XYWH_ABS,
                "category_id": category[obj["name"]],
            }
            for c_idx, obj in enumerate(annotations)
        ]
        dataset.append(record)

    return dataset

In [ ]:
DatasetCatalog.register("data_detection_test", lambda: get_detection_data())
MetadataCatalog.get("data_detection_test").set(thing_classes=list(category.keys()))
metadata = MetadataCatalog.get("data_detection_test")

In [ ]:
cfg = get_cfg()
cfg.OUTPUT_DIR = OUTPUT_DIR
cfg.merge_from_file(
    model_zoo.get_config_file("COCO-Detection/faster_rcnn_X_101_32x8d_FPN_3x.yaml")
)
cfg.DATASETS.TEST = ("data_detection_test",)
cfg.TEST.EVAL_PERIOD = 100
cfg.DATALOADER.NUM_WORKERS = 4
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
    "COCO-Detection/faster_rcnn_X_101_32x8d_FPN_3x.yaml"
)
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 512
cfg.MODEL.ROI_HEADS.NUM_CLASSES = len(category.keys())
cfg.SOLVER.IMS_PER_BATCH = 1

In [ ]:
cfg.MODEL.WEIGHTS = os.path.join(
    cfg.OUTPUT_DIR, "model_final.pth"
)  # path to the model we just trained
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.6  # set a custom testing threshold
predictor = DefaultPredictor(cfg)

In [ ]:
dataset_dicts = get_detection_data()
for d in random.sample(dataset_dicts, 2):
    im = cv2.imread(d["file_name"])
    outputs = predictor(im)
    v_pred = Visualizer(im[:, :, ::-1], metadata=metadata, scale=0.5)
    v_gt = Visualizer(im[:, :, ::-1], metadata=metadata, scale=0.5)
    out_pred = v_pred.draw_instance_predictions(outputs["instances"].to("cpu"))
    out_gt = v_gt.draw_dataset_dict(d)

    concat_image = np.concatenate(
        (out_pred.get_image()[:, :, ::-1], out_gt.get_image()[:, :, ::-1]), axis=1
    )
    plt.figure(figsize=(12, 8))
    plt.imshow(concat_image)

## Inference on single image

In [ ]:
img_path = "test/09WMV85VAMM_tap_007.png"
im = cv2.imread(img_path)
outputs = predictor(im)
v = Visualizer(im[:, :, ::-1], metadata=metadata, scale=0.5)
out = v.draw_instance_predictions(outputs["instances"].to("cpu"))
cv2.imwrite("output.png", out.get_image()[:, :, ::-1])
plt.figure(figsize=(12, 8))
plt.imshow(out.get_image()[:, :, ::-1])

## Evaluation

In [ ]:
"""
# Use COCOEvaluator and build_detection_train_loader
# You can save the output predictions using inference_on_dataset
"""
TEST_OUTPUT_DIR = "{}/test_output".format(BASE_DIR)
os.makedirs(TEST_OUTPUT_DIR, exist_ok=True)

from detectron2.data import build_detection_test_loader
from detectron2.evaluation import COCOEvaluator, inference_on_dataset

evaluator = COCOEvaluator("data_detection_test", output_dir=TEST_OUTPUT_DIR)
test_loader = build_detection_test_loader(cfg, "data_detection_test")
print(inference_on_dataset(predictor.model, test_loader, evaluator))

## Get and Draw False Negatives

In [ ]:
import cv2
import numpy as np
from detectron2.data import MetadataCatalog
from detectron2.structures import Boxes, BoxMode
from detectron2.utils.visualizer import Visualizer


def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interArea = max(0, xB - xA + 1) * max(0, yB - yA + 1)
    boxAArea = (boxA[2] - boxA[0] + 1) * (boxA[3] - boxA[1] + 1)
    boxBArea = (boxB[2] - boxB[0] + 1) * (boxB[3] - boxB[1] + 1)

    iou = interArea / float(boxAArea + boxBArea - interArea)
    return iou


counter = 0
total = 0
category_count = {k: 0 for k in category.keys()}
fn_names = {}
for d in dataset_dicts:
    im = cv2.imread(d["file_name"])
    outputs = predictor(im)
    pred_boxes = outputs["instances"].pred_boxes.tensor.cpu().numpy()

    if not pred_boxes.any():
        fn_names[d["file_name"]] = 0.68

    for ann in d["annotations"]:
        gt_box = BoxMode.convert(ann["bbox"], BoxMode.XYWH_ABS, BoxMode.XYXY_ABS)
        ious = [compute_iou(gt_box, pred_box) for pred_box in pred_boxes]

        if not ious or max(ious) < 0.5:
            v_pred = Visualizer(im[:, :, ::-1], metadata=metadata, scale=0.5)
            v_gt = Visualizer(im[:, :, ::-1], metadata=metadata, scale=0.5)
            out_pred = v_pred.draw_instance_predictions(outputs["instances"].to("cpu"))
            out_gt = v_gt.draw_dataset_dict(d)
            concat_image = np.concatenate(
                (out_pred.get_image()[:, :, ::-1], out_gt.get_image()[:, :, ::-1]),
                axis=1,
            )
            plt.figure(figsize=(12, 8))
            plt.imshow(concat_image)
            plt.title(f"False Negative for {d['file_name']}")
            counter += 1
            category_count[list(category.keys())[ann["category_id"]]] += 1
        total += 1

In [ ]:
print(
    f"The {(counter / total) * 100}% that were not detected based on Average Precision with an Intersection over Union (IoU) of 0.50: {counter}/{total} object instances"
)
print("False Negative counts by category:")
for cat, count in category_count.items():
    print(f"{cat}: {count}")

## Get threshold for each FN

In [ ]:
for d in dataset_dicts:
    if d["file_name"] in fn_names:
        found = False
        while found == False:
            cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = fn_names[d["file_name"]]
            predictor = DefaultPredictor(cfg)
            im = cv2.imread(d["file_name"])
            outputs = predictor(im)
            pred_boxes = outputs["instances"].pred_boxes.tensor.cpu().numpy()
            if pred_boxes.any() or fn_names[d["file_name"]] < 0:
                found = True
            fn_names[d["file_name"]] -= 0.02

In [ ]:
for d in dataset_dicts:
    if d["file_name"] in fn_names:
        cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = fn_names[d["file_name"]]
        predictor = DefaultPredictor(cfg)
        im = cv2.imread(d["file_name"])
        outputs = predictor(im)
        v_pred = Visualizer(im[:, :, ::-1], metadata=metadata, scale=0.5)
        v_gt = Visualizer(im[:, :, ::-1], metadata=metadata, scale=0.5)
        out_pred = v_pred.draw_instance_predictions(outputs["instances"].to("cpu"))
        out_gt = v_gt.draw_dataset_dict(d)
        concat_image = np.concatenate(
            (out_pred.get_image()[:, :, ::-1], out_gt.get_image()[:, :, ::-1]), axis=1
        )
        cv2_imshow(concat_image)

## Get best threshold

In [ ]:
best_ap50 = -1
best_threshold = -1

for threshold in np.arange(0.1, 1.0, 0.05):
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = float(threshold)
    predictor = DefaultPredictor(cfg)
    evaluator = COCOEvaluator("data_detection_test", output_dir=TEST_OUTPUT_DIR)
    metrics = inference_on_dataset(predictor.model, test_loader, evaluator)
    ap50 = metrics["bbox"]["AP50"]
    if ap50 > best_ap50:
        best_ap50 = ap50
        best_threshold = threshold

print(f"Best AP50: {best_ap50} at threshold: {best_threshold}")